# Sentiment Analysis Pipeline

This notebook applies NLP sentiment models to the collected social media data
and produces per-tweet sentiment scores for downstream correlation analysis.

---

## Goal

Map each tweet to a sentiment score (positive / negative / neutral) using:

- **FinBERT** (`ProsusAI/finbert`) — finance-domain BERT model, state of the art for financial text  
- **VADER** (comparison baseline) — fast lexicon-based approach, no GPU required

The two approaches will be compared and their scores aligned to daily price data.

## Expected Output

A sentiment-annotated dataset saved to `data/processed/tweets_with_sentiment.parquet`  
with columns: `tweet_id`, `symbol`, `posted_at`, `sentiment_label`, `sentiment_score`.

This file is the **required input** for `03_correlation_modeling.ipynb`.

## Planned Pipeline

1. Load tweets from SQLite — filter to English, deduplicate, restrict to Focus Tickers  
2. Batch-inference with **FinBERT** (`transformers.pipeline`)  
3. **VADER** scores as baseline for comparison  
4. Compute daily average sentiment per ticker  
5. Save to `data/processed/tweets_with_sentiment.parquet`  
6. Visualise sentiment distribution per ticker and over time

## Prerequisites

- `data/webmining.db` populated (crawler must have run)
- `data/raw/prices/prices.csv` up to date (`src/finance/price_fetcher.py`)
- `01_data_quality_eda.ipynb` reviewed — language distribution and coverage known
- GPU recommended for FinBERT batch inference (CPU fallback works, ~10× slower)

## Status

> ⏳ **Waiting for more data** — implement once sufficient tweet volume is available  
> (target: ≥ 500 English tweets per Focus Ticker for reliable sentiment distributions)

In [ ]:
# TODO: implement after sufficient data is available
#
# Suggested pipeline skeleton:
#
# import sqlite3, pandas as pd
# from pathlib import Path
# from transformers import pipeline
# from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
#
# DB_PATH   = Path('..') / 'data' / 'webmining.db'
# OUT_PATH  = Path('..') / 'data' / 'processed' / 'tweets_with_sentiment.parquet'
# OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
#
# # 1. Load English tweets for Focus Tickers
# con = sqlite3.connect(DB_PATH)
# tweets = pd.read_sql('SELECT tweet_id, symbol, posted_at, content FROM tweets', con)
# tweets = tweets[tweets['lang'] == 'en'].drop_duplicates('tweet_id')
#
# # 2. FinBERT inference
# finbert = pipeline('text-classification', model='ProsusAI/finbert', truncation=True)
# results = finbert(tweets['content'].tolist(), batch_size=32)
# tweets['finbert_label'] = [r['label'] for r in results]
# tweets['finbert_score'] = [r['score'] for r in results]
#
# # 3. VADER baseline
# analyzer = SentimentIntensityAnalyzer()
# tweets['vader_compound'] = tweets['content'].apply(
#     lambda t: analyzer.polarity_scores(str(t))['compound']
# )
#
# # 4. Save
# tweets.to_parquet(OUT_PATH, index=False)
# print(f'Saved {len(tweets):,} rows to {OUT_PATH}')